In [1]:
RUN_NAME = "localiser_run4c_vgg16enc_patch_reg"
ENCODER = "vgg16"
EPOCHS = 120
SEED = 28                  # as run 4, so the patch plans are the same draws; only the additions differ

MERGED_STORE = "/kaggle/input/datasets/tataruteodor/ddsm-localiser-tensors-merged"   # <- adjust to the attached dataset path
LESION_STORE = "/kaggle/input/datasets/tataruteodor/localiser-tensors"          # <- adjust
SRC_DATASET  = "/kaggle/input/datasets/tataruteodor/src-l4c"                    # <- adjust (contains src/)
OUT = f"/kaggle/working/{RUN_NAME}"

In [2]:
import os, shutil, sys, json
import tensorflow as tf

if os.path.exists("/kaggle/working/src"):
    shutil.rmtree("/kaggle/working/src")
shutil.copytree(os.path.join(SRC_DATASET, "src"), "/kaggle/working/src")
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

print("TF", tf.__version__, "| GPUs", tf.config.list_physical_devices("GPU"))
tf.keras.mixed_precision.set_global_policy("float32")
assert tf.keras.mixed_precision.global_policy().name == "float32", "run 4c is float32 by design"

from src import config, localise
from src.train_localiser_l4 import r4c_kwargs
R4C = r4c_kwargs()
print(json.dumps({k: v for k, v in R4C.items()}, indent=2, default=str))
m = localise.build_unet_pretrained(ENCODER, decoder_dropout=R4C["decoder_dropout"], decoder_l2=R4C["decoder_l2"])
print(f"{m.name}: {m.count_params():,} params (same layout as run 4); "
      f"dropout layers {[l.name for l in m.layers if l.name.endswith('_drop')]}")
del m
for d in (MERGED_STORE, LESION_STORE):
    print(d, sorted(os.listdir(d)))

TF 2.20.0 | GPUs [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
{
  "decoder_dropout": 0.3,
  "decoder_l2": 0.0001,
  "centred_frac": 0.3,
  "centred_stochastic": true,
  "aug": {
    "alpha": [
      3.0,
      6.0
    ],
    "sigma": [
      32.0,
      64.0
    ],
    "gamma": [
      0.7,
      1.4
    ],
    "brightness": 0.15,
    "contrast": 0.15,
    "noise_sd": 0.01
  }
}


I0000 00:00:1788889003.475682      24 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
unet_vgg16_localiser: 31,410,497 params (same layout as run 4); dropout layers ['bottleneck_drop', 'dec5_drop', 'dec4_drop', 'dec3_drop', 'dec2_drop', 'dec1_drop']
/kaggle/input/datasets/tataruteodor/ddsm-localiser-tensors-merged ['train_images.npy', 'train_masks.npy', 'train_meta.csv', 'val_images.npy', 'val_masks.npy', 'val_meta.csv']
/kaggle/input/datasets/tataruteodor/localiser-tensors ['test_images.npy', 'test_masks.npy', 'test_meta.csv', 'train_images.npy', 'train_masks.npy', 'train_meta.csv', 'val_images.npy', 'val_masks.npy', 'val_meta.csv']


In [3]:
from src.train_localiser_l4 import main
manifest = main(MERGED_STORE, LESION_STORE, OUT,
                encoder=ENCODER, whole_image=False,
                epochs=EPOCHS, run_name=RUN_NAME, seed=SEED, **R4C)
print(json.dumps({k: manifest[k] for k in (
    "run", "regime", "params", "steps_per_epoch", "epochs_run", "first_nan_epoch",
    "best_epoch", "val_hard_iou_at_best_epoch", "val_dice_at_best_epoch",
    "val_loss_at_best_epoch", "train_hard_iou_at_best_epoch", "best_valloss_epoch",
    "max_val_hard_iou_any_epoch", "lesion_centred_frac", "decoder_dropout", "decoder_l2",
    "wall_time_s") if k in manifest}, indent=2))

[l4] train store (1146, 1024, 576) float16 (merged, one row per mammogram) | val store (243, 1024, 576) float16 (per lesion) | output prior 0.00497
[l4] unet_vgg16_localiser: 31,410,497 params (14,714,688 encoder at 0.0001, 16,685,825 decoder at 0.001) | regime patch512 | batch 8 | 287 steps/epoch x 120
Epoch 1/120


2026-09-08 17:39:59.502152: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv (f32[32,96,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,96,512,512]{3,2,1,0}, f32[8,32,512,512]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2026-09-08 17:40:00.740472: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 2.238454328s
Trying algorithm eng0{} for conv (f32[32,96,3,3]{3,2,1,0}, u8[0]{0}) custom-call(f32[8,96,512,512]{3,2,1,0}, f32[8,32,512,512]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardFilter", backend_config={"operation_queu

287/287 ━━━━━━━━━━━━━━━━━━━━ 0s 805ms/step - dice_coefficient: 0.0768 - hard_iou: 0.0644 - loss: 1.1070

2026-09-08 17:44:15.241039: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 17:44:15.457746: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 17:44:33.512825: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 17:44:33.714585: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-08 17:44:34.711943: E external/local_xla/xla/stream_


Epoch 1: val_hard_iou improved from None to 0.17541, saving model to /kaggle/working/localiser_run4c_vgg16enc_patch_reg/best.weights.h5

Epoch 1: finished saving model to /kaggle/working/localiser_run4c_vgg16enc_patch_reg/best.weights.h5
287/287 ━━━━━━━━━━━━━━━━━━━━ 375s 1s/step - dice_coefficient: 0.1820 - hard_iou: 0.1502 - loss: 0.9273 - val_dice_coefficient: 0.2702 - val_hard_iou: 0.1754 - val_loss: 0.6928 - lesion_centred_frac: 0.3089 - fg_patch_frac: 0.9114 - aug_elastic_alpha_mean: 4.5018 - aug_elastic_sigma_mean: 48.2332 - aug_gamma_mean: 1.0481 - aug_brightness_mean: -0.0035 - aug_contrast_mean: 1.0019 - aug_mean_disp_px_mean: 4.5016 - lr_encoder: 2.0070e-05 - lr_decoder: 2.0070e-04
Epoch 2/120
287/287 ━━━━━━━━━━━━━━━━━━━━ 0s 786ms/step - dice_coefficient: 0.3311 - hard_iou: 0.2621 - loss: 0.5663
Epoch 2: val_hard_iou did not improve from 0.17541
287/287 ━━━━━━━━━━━━━━━━━━━━ 238s 830ms/step - dice_coefficient: 0.3454 - hard_iou: 0.2704 - loss: 0.5114 - val_dice_coefficient: 0

In [4]:
import pandas as pd
h = pd.read_csv(f"{OUT}/history.csv")
cols = [c for c in h.columns if c.startswith("aug_") or c in ("lesion_centred_frac", "fg_patch_frac")]
print(h[["epoch", "dice_coefficient", "hard_iou", "val_dice_coefficient", "val_hard_iou"] + cols]
      .describe().loc[["mean", "min", "max"]].round(4).T)
print("train-val gap at best epoch: dice %.3f vs %.3f | hard IoU %.3f vs %.3f" % (
    h.loc[h.val_hard_iou.idxmax(), "dice_coefficient"], h.loc[h.val_hard_iou.idxmax(), "val_dice_coefficient"],
    h.loc[h.val_hard_iou.idxmax(), "hard_iou"], h.loc[h.val_hard_iou.idxmax(), "val_hard_iou"]))

                           mean      min       max
epoch                   59.5000   0.0000  119.0000
dice_coefficient         0.5845   0.1820    0.7216
hard_iou                 0.4434   0.1502    0.5607
val_dice_coefficient     0.4230   0.0755    0.4832
val_hard_iou             0.3345   0.0668    0.3935
aug_brightness_mean      0.0000  -0.0052    0.0040
aug_contrast_mean        1.0001   0.9949    1.0045
aug_elastic_alpha_mean   4.5032   4.4512    4.5480
aug_elastic_sigma_mean  47.9685  47.3357   48.4375
aug_gamma_mean           1.0502   1.0381    1.0624
aug_mean_disp_px_mean    4.5030   4.4511    4.5479
fg_patch_frac            0.9135   0.9005    0.9236
lesion_centred_frac      0.3011   0.2757    0.3229
train-val gap at best epoch: dice 0.580 vs 0.481 | hard IoU 0.427 vs 0.394


In [5]:
from pathlib import Path
from src import localise_eval
localise_eval.TENSORS = Path(LESION_STORE)
model = localise_eval._load_model(f"{OUT}/best.weights.h5", config.LOC_BASE_FILTERS,
                                  config.LOC_DEPTH, encoder=ENCODER)
df = localise_eval.evaluate_split(model, "val")
rep = localise_eval.report(df, "val")
df.to_csv(f"{OUT}/localiser_boxes_val.csv", index=False)
json.dump([rep], open(f"{OUT}/localiser_metrics_val.json", "w"), indent=2)
print({k: round(rep[k], 4) for k in ("box_iou_inclusive_mean", "detection_rate",
                                     "box_iou_mean_given_detected", "detect_at_iou50", "dice_mean")})

[localise_eval] rebuilt vgg16-encoder U-Net (31,410,497 params) from /kaggle/working/localiser_run4c_vgg16enc_patch_reg/best.weights.h5


2026-09-09 01:30:22.067197: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 01:30:22.292544: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 01:30:54.895925: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 01:30:55.103856: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-09-09 01:30:56.784330: E external/local_xla/xla/stream_


=== val (n=243, scored=243, no GT=0) ===
  detection_rate                 0.8971
  fallback_rate                  0.1029
  mask_iou_mean                  0.3934
  dice_mean                      0.4945
  box_iou_inclusive_mean         0.4188
  box_iou_inclusive_median       0.5597
  box_iou_mean_given_detected    0.4668
  box_iou_mammogram_mean         0.4636
  detect_at_iou50                0.5391
  detect_at_iou30                0.6091
  centroid_hit_rate              0.6461
  box_iou_any_inclusive_mean     0.4530
  pct_rows_on_multilesion_images 0.1235
  box_iou_matched_k3_inclusive_mean 0.4854
  detect_at_iou50_matched_k3     0.6255
  detection_rate_matched_k3      0.7654
  n_rows_matched_box_differs_from_single 61.0000
  n_images                       225.0000
  dice_image_mean_inclusive      0.5223
  dice_image_mean_detected       0.5818
{'box_iou_inclusive_mean': 0.4188, 'detection_rate': 0.8971, 'box_iou_mean_given_detected': 0.4668, 'detect_at_iou50': 0.5391, 'dice_mean': 0.49

In [6]:
# Download convention on the laptop:
#   outputs/weights/localiser_run4c_vgg16enc_patch_reg/{best.weights.h5, best_valloss.weights.h5, manifest.json, history.csv}
# then, on the laptop:
#   python -m src.localise_eval --encoder vgg16 --weights outputs/weights/localiser_run4c_vgg16enc_patch_reg/best.weights.h5 \
#       --splits val train --out-dir outputs/results/localiser_run4c_eval
print(sorted(os.listdir(OUT)))
shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", OUT)
print(f"-> /kaggle/working/{RUN_NAME}.zip")

['best.weights.h5', 'best_valloss.weights.h5', 'breast_boxes_train.npy', 'history.csv', 'localiser_boxes_val.csv', 'localiser_metrics_val.json', 'manifest.json']
-> /kaggle/working/localiser_run4c_vgg16enc_patch_reg.zip
